# DeepShield: budget-matched arms (joint16, shield4_mist4)

`shield_mist16` (8/255 shield, then 8/255 Mist) can move a pixel up to 16/255; `joint8` only 8/255.
This runs the two arms that match them on the four pilot people (hugo_chavez, jeb_bush, jennifer_lopez, nicole_kidman):

- **joint16**: one PGD with both objectives in 16/255, against `shield_mist16`.
- **shield4_mist4**: 4/255 of shield, then 4/255 of Mist, against `joint8`.

Put `code.zip` and `inputs.zip` (made by `experiments/lora_defense/colab_bundle_budget.sh`) in **MyDrive/deepshield_budget/**,
choose a T4 GPU runtime and run every cell. If the runtime disconnects, run every cell again: every step resumes from Drive.
Results land in `MyDrive/deepshield_budget/work/results/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
D = '/content/drive/MyDrive/deepshield_budget'
for name in ('code.zip', 'inputs.zip'):
    assert os.path.exists(f'{D}/{name}'), f'{name} is not in {D}'
!rm -rf /content/project-research && mkdir -p /content/project-research
!unzip -q -o {D}/code.zip -d /content/project-research
# The inputs go onto Drive once: later steps add to them (shield4_512, the manifest), so never again.
if not os.path.exists(f'{D}/input/.unpacked'):
    !mkdir -p {D}/input && unzip -q -o {D}/inputs.zip -d {D}/input && touch {D}/input/.unpacked
!ls {D}/input/v3 {D}/input/v3/protected

In [ ]:
!pip install -q -e "/content/project-research[face,torch,diffusion,video]" "peft>=0.11" uv
# peft refuses an older torchao, and nothing here uses it.
!pip uninstall -q -y torchao || true
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content/project-research
!DS_INPUT={D}/input DS_WORK={D}/work bash experiments/lora_defense/colab_run_budget.sh 2>&1 | tee -a {D}/run.log

### Scoring on a CPU runtime

Both scorers run ONNX Runtime on the CPU only, and an hour of an idle T4 gets the runtime reclaimed, so `colab_run_budget.sh` stops after generation.
Choose **Runtime → Change runtime type → CPU**, run the cells above up to the install (skip the GPU run cell), then this one. It resumes too: a report already on Drive is not redone.

In [ ]:
%cd /content/project-research
!DS_INPUT={D}/input DS_WORK={D}/work bash experiments/lora_defense/colab_score_budget.sh 2>&1 | tee -a {D}/run.log

In [ ]:
import json
R = f'{D}/work/results'
photos = json.load(open(f'{R}/budget_photos.json'))
lora = json.load(open(f'{R}/lora/lora_defense.json'))
print('swap, owner ranked first (ArcFace) / with a face:')
for condition, s in photos['swaps'].items():
    print(f"  {condition:28s} {s['arcface_owner_first']:3d} / {s['with_a_face']}")
print('quality:', json.dumps(photos['quality_vs_clean_at_512_bicubic'], indent=1))
print('watermark read:', json.dumps(photos['watermark_read'], indent=1))
print('LoRA, person ranked first (ArcFace) / with a face:')
for variant, s in lora['variants'].items():
    print(f"  {variant:28s} {s['arcface']['person_first']:3d} / {s['with_a_face']}  high-confidence {s['arcface']['above_high_confidence']}")

In [ ]:
# One zip to bring back: results, generated images, protected photos, logs.
!cd {D}/work && zip -qr {D}/results_budget.zip results generated protected log.txt mist_*.log
!ls -la {D}/results_budget.zip